In [ ]:
# =========================================================
# RoBERTa + BiLSTM Hybrid for Binary Sentiment Classification
# =========================================================

import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt

# -------------------------
# 1. Reproducibility
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -------------------------
# 2. Load dataset
# -------------------------
FILE_PATH = "binary_standard.csv"   # change this only
df = pd.read_csv(FILE_PATH)

print("Dataset shape:", df.shape)
print(df.head())
print(df["label"].value_counts())

# -------------------------
# 3. Keep only needed columns
# -------------------------
# Use text only first for clean RoBERTa+BiLSTM baseline
# If your dataset has "Reviews", keep that
# If it has "text", rename it

if "Reviews" in df.columns:
    df = df[["Reviews", "label"]].copy()
    df = df.rename(columns={"Reviews": "text"})
elif "text" in df.columns:
    df = df[["text", "label"]].copy()
else:
    raise ValueError("No text column found. Expected 'Reviews' or 'text'.")

df["text"] = df["text"].astype(str)

# -------------------------
# 4. Train-test split
# -------------------------
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# -------------------------
# 5. Tokenizer
# -------------------------
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 180

# -------------------------
# 6. Dataset class
# -------------------------
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "label": torch.tensor(label, dtype=torch.long)
        }

train_dataset = SentimentDataset(
    texts=train_df["text"].tolist(),
    labels=train_df["label"].tolist(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = SentimentDataset(
    texts=test_df["text"].tolist(),
    labels=test_df["label"].tolist(),
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# -------------------------
# 7. RoBERTa + BiLSTM Model
# -------------------------
class RobertaBiLSTM(nn.Module):
    def __init__(self, model_name, hidden_dim, lstm_layers, dropout, output_dim):
        super(RobertaBiLSTM, self).__init__()

        self.roberta = AutoModel.from_pretrained(model_name)
        roberta_hidden_size = self.roberta.config.hidden_size  # 768 for roberta-base

        self.lstm = nn.LSTM(
            input_size=roberta_hidden_size,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, input_ids, attention_mask):
        roberta_outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = roberta_outputs.last_hidden_state   # [batch, seq_len, 768]

        lstm_out, (hidden, cell) = self.lstm(sequence_output)

        forward_hidden = hidden[-2, :, :]
        backward_hidden = hidden[-1, :, :]
        combined = torch.cat((forward_hidden, backward_hidden), dim=1)

        out = self.dropout(combined)
        out = self.fc(out)

        return out

HIDDEN_DIM = 128
LSTM_LAYERS = 1
DROPOUT = 0.3
OUTPUT_DIM = 2

model = RobertaBiLSTM(
    model_name=MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    lstm_layers=LSTM_LAYERS,
    dropout=DROPOUT,
    output_dim=OUTPUT_DIM
).to(device)

print(model)

# -------------------------
# 8. Loss and optimizer
# -------------------------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -------------------------
# 9. Training loop
# -------------------------
EPOCHS = 3

train_losses = []
test_losses = []

best_val_loss = float("inf")
patience = 2
counter = 0

for epoch in range(EPOCHS):
    model.train()
    running_train_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    model.eval()
    running_test_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            running_test_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_test_loss = running_test_loss / len(test_loader)
    test_losses.append(avg_test_loss)

    acc = accuracy_score(all_labels, all_preds)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Test Loss : {avg_test_loss:.4f}")
    print(f"Accuracy  : {acc:.4f}")
    print("-" * 40)

    # Early stopping
    if avg_test_loss < best_val_loss:
        best_val_loss = avg_test_loss
        counter = 0
        torch.save(model.state_dict(), "best_roberta_bilstm.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

# Load best model
model.load_state_dict(torch.load("best_roberta_bilstm.pt"))

# -------------------------
# 10. Final evaluation
# -------------------------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average="weighted", zero_division=0
)

print("\nFinal RoBERTa + BiLSTM Results")
print("Accuracy :", acc)
print("Precision:", precision)
print("Recall   :", recall)
print("F1       :", f1)

print("\nClassification Report")
print(classification_report(
    all_labels,
    all_preds,
    target_names=["negative", "positive"],
    zero_division=0
))

# -------------------------
# 11. Plot loss curves
# -------------------------
plt.figure(figsize=(8,5))
plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss")
plt.plot(range(1, len(test_losses)+1), test_losses, label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("RoBERTa + BiLSTM Loss Curve")
plt.legend()
plt.show()